# Validazione holdout end-to-end degli Instance Graph

Questo notebook verifica la generalizzazione della pipeline principale su process execution non utilizzate per apprendere le causal relations.

## Impostazione metodologica

L'OCEL strutturale viene separato per componenti connesse, senza eventi oppure oggetti condivisi. L'OC-DFG e le causal relations vengono derivati esclusivamente dal training.

Le relazioni full-log vengono calcolate successivamente e utilizzate soltanto come baseline valutativa per confrontare la topologia dei grafi.

In [1]:
import pandas as pd

from ocpm_partial_order.config import MAIN_DATASET_DB
from ocpm_partial_order.conformance import (
    evaluate_instance_graph_holdout,
)
from ocpm_partial_order.io.ocel_loader import (
    load_ocel2_sqlite,
)

## Esecuzione della pipeline

Le soglie restano fissate ai valori adottati nel progetto: dependency `0.90`, supporto relativo `0.05` e self-loop `0.90`.

In [2]:
ocel = load_ocel2_sqlite(MAIN_DATASET_DB)

evaluation = evaluate_instance_graph_holdout(ocel)

print('Valutazione completata.')

Valutazione completata.


## Separazione senza leakage

In [3]:
split = evaluation.split

split_summary = pd.DataFrame(
    [
        {
            'componenti totali': split.component_count,
            'componenti training': split.training_component_count,
            'componenti test': split.test_component_count,
            'eventi training': split.training_event_count,
            'eventi test': split.test_event_count,
            'oggetti training': split.training_object_count,
            'oggetti test': split.test_object_count,
            'rapporto training': round(
                split.effective_training_ratio,
                4,
            ),
            'eventi condivisi': split.shared_event_count,
            'oggetti condivisi': split.shared_object_count,
        }
    ]
)

split_summary

,componenti totali,componenti training,componenti test,eventi training,eventi test,oggetti training,oggetti test,rapporto training,eventi condivisi,oggetti condivisi
0,68,44,24,16547,4461,8531,2256,0.7877,0,0


## Relazioni apprese

Il confronto con la baseline full-log serve soltanto a misurare se il training ha recuperato lo stesso insieme di relazioni. La costruzione holdout non utilizza la baseline.

In [4]:
relation_summary = pd.DataFrame(
    [
        {
            'relazioni training': len(
                evaluation.training_relations
            ),
            'relazioni baseline': len(
                evaluation.reference_relations
            ),
            'mancanti nel training': len(
                evaluation.missing_training_relations
            ),
            'aggiuntive nel training': len(
                evaluation.additional_training_relations
            ),
            'insieme esatto': evaluation.exact_relation_set,
        }
    ]
)

relation_summary

,relazioni training,relazioni baseline,mancanti nel training,aggiuntive nel training,insieme esatto
0,26,26,0,0,True


## Risultati degli Instance Graph

Gli ordini strutturalmente contaminati vengono esclusi perche non rispettano la definizione order-centred del prototipo. Non sono conteggiati come grafi falliti.

In [5]:
graph_results = pd.DataFrame(
    [
        {
            'ordine': result.order_id,
            'eventi': result.event_count,
            'nodi': result.node_count,
            'archi': result.edge_count,
            'DAG': result.is_dag,
            'connesso': result.is_weakly_connected,
            'copertura': result.covers_all_events,
            'ridotto': result.is_transitively_reduced,
            'topologia esatta': result.exact_topology,
        }
        for result in evaluation.results
    ]
)

print('Ordini nel test:', evaluation.test_order_count)
print('Ordini esclusi:', evaluation.excluded_order_count)
print('Ordini valutati:', evaluation.evaluated_order_count)
print('Grafi validi:', evaluation.valid_graph_count)
print('Topologie esatte:', evaluation.exact_topology_count)

graph_results

Ordini nel test: 436
Ordini esclusi: 427
Ordini valutati: 9
Grafi validi: 9
Topologie esatte: 9


,ordine,eventi,nodi,archi,DAG,connesso,copertura,ridotto,topologia esatta
0,o-991144,11,11,14,True,True,True,True,True
1,o-991284,8,8,8,True,True,True,True,True
2,o-991324,11,11,14,True,True,True,True,True
3,o-991520,11,11,12,True,True,True,True,True
4,o-991630,9,9,10,True,True,True,True,True
5,o-991686,10,10,12,True,True,True,True,True
6,o-991749,7,7,6,True,True,True,True,True
7,o-991925,11,11,14,True,True,True,True,True
8,o-991982,9,9,10,True,True,True,True,True


## Verifiche automatiche

In [6]:
assert evaluation.split.shared_event_count == 0
assert evaluation.split.shared_object_count == 0
assert len(evaluation.training_relations) == 26
assert evaluation.exact_relation_set
assert evaluation.test_order_count == 436
assert evaluation.excluded_order_count == 427
assert evaluation.evaluated_order_count == 9
assert evaluation.valid_graph_count == 9
assert evaluation.exact_topology_count == 9
assert evaluation.all_evaluated_graphs_valid
assert evaluation.all_topologies_exact

print('Tutte le verifiche sono state superate.')

Tutte le verifiche sono state superate.


## Interpretazione e limiti

Le 26 causal relations apprese esclusivamente dal training coincidono con la baseline full-log. Tutte le nove process execution ammissibili del test producono Instance Graph DAG, connessi, completi e transitivamente ridotti; le nove topologie coincidono esattamente con la baseline.

I 427 ordini esclusi sono collegati strutturalmente ad altri ordini e ricadono fuori dalla definizione order-centred adottata. Il risultato dimostra la generalizzazione sui casi ammissibili del test, ma non rappresenta un replay globalmente sincronizzato sulla OCPN.